# AraContract RAG + Frontend on Google Colab

يشغّل هذا الدفتر RAG الخاص بالمشروع باستخدام **Ollama** على Colab، وينشر API عبر **ngrok**، ثم يعرض واجهة عربية جاهزة للتجربة. يعتمد مسار الاتصال في `Connect-rag-with-colab.pdf` ويستخدم endpoints المشروع الفعلية.

> فعّل GPU من: `Runtime → Change runtime type → T4 GPU`. رابط ngrok عام ومؤقت؛ لا ترفع عقوداً سرية إليه.

## 1. تحميل المشروع وتثبيت تبعيات RAG

In [ ]:
from pathlib import Path
import os, subprocess, time

REPO_DIR = Path('/content/AraContract-Analyzer')
if not REPO_DIR.exists():
    !git clone https://github.com/eliasnadder/AraContract-Analyzer.git {REPO_DIR}
%cd {REPO_DIR}
assert (REPO_DIR / 'backend/app/routers/rag.py').exists()

%pip -q install -U fastapi 'uvicorn[standard]' pydantic-settings httpx pyngrok nest-asyncio qdrant-client sentence-transformers langchain-huggingface langchain-text-splitters huggingface-hub transformers

## 2. تنزيل نماذج الاسترجاع

يستخدم الـ backend مسارات محلية تحت `backend/models_local`، لذا نحفظ snapshots فيها.

In [ ]:
from huggingface_hub import snapshot_download

models_dir = REPO_DIR / 'backend/models_local'
for model_id, folder in {
    'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2': 'paraphrase-multilingual-MiniLM-L12-v2',
    'cross-encoder/ms-marco-MiniLM-L-6-v2': 'cross-encoder-ms-marco-MiniLM-L-6-v2',
}.items():
    snapshot_download(repo_id=model_id, local_dir=str(models_dir / folder))

## 3. تشغيل Ollama

النموذج الافتراضي `qwen2.5:3b` مناسب كبداية على T4. غيّره إلى نموذج أصغر إذا كانت الذاكرة محدودة.

In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh
ollama_process = subprocess.Popen(['ollama', 'serve'])
time.sleep(5)

OLLAMA_MODEL = 'qwen2.5:3b'
!ollama pull {OLLAMA_MODEL}
os.environ.update({
    'LLM_PROVIDER': 'ollama',
    'OLLAMA_BASE_URL': 'http://127.0.0.1:11434',
    'OLLAMA_MODEL': OLLAMA_MODEL,
    'QDRANT_MODE': 'memory',
})

## 4. تشغيل RAG API

يستعمل `colab_rag_server.py` نفس RAG router في المشروع، مع CORS للواجهة. لا يحتاج إعداد Firebase الخاص ببقية المسارات المحمية.

In [ ]:
api_process = subprocess.Popen([
    'uvicorn', 'colab_rag_server:app', '--host', '0.0.0.0', '--port', '8000'
], cwd=REPO_DIR / 'backend')
time.sleep(5)
print('RAG API: http://127.0.0.1:8000')

## 5. فتح رابط API عام عبر ngrok

أنشئ token مجاني من https://dashboard.ngrok.com/get-started/your-authtoken. لن يظهر token في المخرجات.

In [ ]:
from getpass import getpass
from pyngrok import ngrok
import requests

ngrok.set_auth_token(getpass('ngrok auth token: '))
ngrok.kill()
tunnel = ngrok.connect(8000, bind_tls=True)
PUBLIC_URL = tunnel.public_url.rstrip('/')
API_BASE = f'{PUBLIC_URL}/api/contract/rag'
print('RAG API:', API_BASE)
print('Docs:', f'{PUBLIC_URL}/docs')
requests.get(f'{API_BASE}/status', timeout=30).json()

## 6. اختبار RAG من Python

In [ ]:
clauses = [
    'المادة الأولى: يلتزم الطرف الأول بتقديم الخدمة المتفق عليها خلال خمسة أيام عمل.',
    'المادة الثانية: يلتزم الطرف الثاني بسداد المستحقات خلال ثلاثين يوماً من تاريخ الفاتورة.',
    'المادة الثالثة: يحق للطرف الأول إنهاء العقد دون إشعار مسبق عند إخلال الطرف الثاني بالتزاماته.',
]
ingest = requests.post(f'{API_BASE}/ingest', json={'clauses': clauses}, timeout=180)
ingest.raise_for_status()
SESSION_ID = ingest.json()['session_id']
ask = requests.post(f'{API_BASE}/ask', json={'session_id': SESSION_ID, 'question': 'هل يمكن إنهاء العقد دون إشعار مسبق؟', 'top_k': 3}, timeout=180)
ask.raise_for_status()
ask.json()

## 7. الواجهة الأمامية

الخلية التالية تحقن رابط ngrok في `frontend/rag-client.html` ثم تعرض الواجهة داخل الدفتر. لدمجها في تطبيقك، اضبط `window.__RAG_API_BASE__` قبل تحميل الملف بقيمة `API_BASE`.

In [ ]:
from IPython.display import HTML, display

frontend = (REPO_DIR / 'frontend/rag-client.html').read_text(encoding='utf-8')
frontend = frontend.replace('window.__RAG_API_BASE__ = "";', f'window.__RAG_API_BASE__ = {API_BASE!r};')
display(HTML(frontend))

## 8. إنهاء الجلسة

عند الانتهاء، احذف جلسة RAG لتحرير ذاكرة Qdrant. جلسات Colab وngrok مؤقتة ويجب إعادة تشغيل الخلايا بعد انتهائها.

In [ ]:
# requests.delete(f'{API_BASE}/session/{SESSION_ID}', timeout=30).raise_for_status()
# ngrok.kill()
# api_process.terminate()
# ollama_process.terminate()